# 基于大语言模型的古今异义词判别系统 - 叶丛榕 (202521091010)

## 任务目标
探索如何利用大语言模型（Large Language Model, LLM）实现对汉语词汇“古今异义”现象的自动判别，即判断某一古籍中的词义（古义）在现代汉语中是否发生显著语义偏移、消亡，或仅存在于书面语/古语场景（即便语义一致仍判定为古今异义），最终完成“古今异义（Label 1）”与“语义延续（Label 0）”的二元分类。

## 技术方法
本系统采用**提示工程（Prompt Engineering）驱动的少样本推理范式**，以通义千问（Qwen）为核心模型（兼容OpenAI API接口），构建结构化提示模板：
1. 基础框架：“系统指令 + 少样本示例 + 目标样本”三层结构，明确模型的“语言学家”角色与判定规则；
2. 特征融合：在提示中嵌入多维语言学特征——古义描述、格式化现代义项列表、关键词匹配度、现代义项限制性标签（<书>/<古>/方言/成语）、历史频率衰减模式、词汇生僻度等；
3. 输出约束：强制模型返回JSON格式结果（包含样本ID、推理过程、分类标签），确保结果结构化、可解析；
4. 工程保障：设计数据清洗、重试机制、结果校验等模块，提升系统鲁棒性。

## 最终方案
经多轮提示模板迭代与模型行为分析，确定核心判别流程：
1. 数据预处理：清洗训练集异常值（负数频次），对古今义项做特征工程，生成关键词匹配度、标签特征、衰减模式等辅助信息；
2. 少样本采样：从高置信训练样本（all_freq≥20）中均衡选取正负例，确保示例代表性；
3. 结构化推理：引导模型遵循“语义匹配→标签检查→频率验证”三步判定逻辑，降低随机性（temperature=0）；
4. 批量预测：支持验证集评估与测试集批量预测，输出仅含样本ID和分类标签的标准化结果文件。

## 性能评估
针对无大规模标注测试集的场景，采用“定量+定性”结合的评估策略：
1. 定量评估：将训练集按8:2拆分（演示池+验证集），计算准确率（Accuracy）、分类报告（精确率/召回率/F1），统计混淆矩阵（TP/FP/TN/FN），定位错误样本并保存分析；
2. 定性评估：人工抽样检查模型推理链的逻辑连贯性，分析错误案例的成因（如特征缺失、规则理解偏差），反向优化提示模板与特征工程逻辑。

## 技术路线
1. 数据层：原始语料加载 → 异常值清洗 → 义项格式化 → 多维语言学特征工程（关键词匹配/标签提取/频率衰减/生僻度）
2. 提示层：判定规则定义 → 高置信少样本示例筛选 → 特征嵌入结构化Prompt
3. 推理层：通义千问客户端初始化 → 单样本重试预测 → JSON结果解析与校验
4. 验证层：训练集拆分评估 → 准确率/F1计算 → 错误样本分析与保存
5. 应用层：测试集全量预处理 → 批量预测 → UTF-8编码结果导出（仅ID+标签）

本系统无需传统机器学习的训练过程，通过“数据清洗-特征增强-提示引导”的端到端流程，充分激发大语言模型的语义理解能力，为低资源、高语义复杂度的古今异义判别任务提供了高效、可解释且工程化的解决方案。

In [ ]:
import os
import sys
import json
import time
import pandas as pd
from openai import OpenAI
import asyncio
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import accuracy_score, classification_report
import openpyxl

### 0. 数据异常值处理
#### 功能
- 检查原始训练集文件是否存在，缺失则终止程序
- 校验数据必需列（id/sense/modern_sense_list/label）
- 识别并删除含负数的数值型字段（_freq/_count 结尾列）异常样本
- 保存清洗后的数据到指定路径，返回清洗后的 DataFrame 和保存路径

#### 关键路径
- 原始训练集：./train.xlsx
- 清洗后训练集：./train_cleaned.xlsx

In [ ]:
# -------------------------- 数据异常值处理 --------------------------
CLEANED_TRAIN_PATH = "./train_cleaned.xlsx"
RAW_TRAIN_PATH = "./train.xlsx"
RAW_TEST_PATH = "./test.xlsx"

def load_and_preprocess_data(
    input_path=RAW_TRAIN_PATH,
    output_path=CLEANED_TRAIN_PATH
):
    """
    清洗训练集并保存。
    
    Args:
        input_path (str): 原始训练集路径
        output_path (str): 清洗后保存路径
    
    Returns:
        tuple: (cleaned_df, output_path)
    """
    # 检查输入文件
    if not os.path.exists(input_path):
        print(f"❌ 找不到原始训练集: {os.path.abspath(input_path)}")
        sys.exit(1)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # 加载
    train_df = pd.read_excel(input_path, engine="openpyxl")
    print(f"✅ 加载 {len(train_df)} 条样本")

    # 字段检查
    for col in ["id", "sense", "modern_sense_list", "label"]:
        if col not in train_df.columns:
            raise ValueError(f"缺失列: {col}")

    # 异常值清洗
    numeric_cols = [c for c in train_df.columns if c.endswith('_freq') or c.endswith('_count')]
    def has_negative(row):
        for col in numeric_cols:
            val = row[col]
            if pd.notna(val) and isinstance(val, (int, float)) and val < 0:
                return True
        return False

    mask = ~train_df.apply(has_negative, axis=1)
    removed = len(train_df) - mask.sum()
    if removed > 0:
        print(f"⚠️  删除 {removed} 行异常样本")
        train_df = train_df[mask].reset_index(drop=True)

    # 保存
    train_df.to_excel(output_path, index=False)
    abs_path = os.path.abspath(output_path)
    print(f"💾 已保存清洗后数据至: {abs_path}")

    return train_df, output_path  # 👈 关键：返回路径！

if __name__ == "__main__" or "ipykernel" in sys.modules:
    # 在 Jupyter/Notebook 中自动运行
    print("🚀 开始数据预处理...")
    clean_train_df, CLEANED_TRAIN_PATH_USED = load_and_preprocess_data()

print("✅ 数据预处理模块已定义并执行完成")

### 1. 数据预处理（增强版）
#### 核心功能
1. 格式标准化：将现代义项列表字符串转为易读格式
2. 特征工程：
   - 关键词匹配度：判断古义核心词是否出现在现代义项中（高度/部分/几乎未重合）
   - 限制性标签提取：识别现代义项中的<书>/<古>/方言/成语等标记
   - 历史频率衰减特征：基于朝代频次计算词汇使用重心（古代/居中/近现代）
   - 生僻度：按总频次（all_freq）区分生僻词/常用词
3. 支持训练集/测试集双输入，分别完成预处理

#### 关键函数
- format_modern_sense：格式化现代义项文本
- compute_keyword_match：计算古义-现代义项关键词匹配度
- extract_modern_flags：提取现代义项的限制性标签
- compute_decay_feature：计算历史频率衰减模式
- numeric_to_text：整合所有特征并添加到 DataFrame

In [ ]:
# -------------------------- 1. 数据预处理模块 --------------------------
# 🔑 定义清洗后训练集的固定路径（供本模块和其他模块使用）
CLEANED_TRAIN_PATH = "./train_cleaned.xlsx"
TEST_PATH = "./test.xlsx"  # 如果你需要默认测试集路径


def load_and_preprocess_data(train_path=CLEANED_TRAIN_PATH, test_path=None):
    # ... (前面的文件加载和检查代码保持不变，直到 define format_modern_sense)

    def format_modern_sense(sense_str):
        if pd.isna(sense_str) or sense_str == "": return "无现代义项"
        # 简单清洗，保留列表结构感
        return str(sense_str).replace("['", "1. ").replace("']", "").replace("', '", "; ").replace("['", "").replace("']", "")

    # 【新增】计算字面匹配度：古义的词是否直接出现在现代义项中？
    def compute_keyword_match(row):
        target_sense = str(row['sense'])
        modern_senses = str(row['modern_sense_list'])
        
        # 提取古义中的核心词（去掉标点和虚词，简单版）
        keywords = [k for k in target_sense if k not in "，。的等地得"]
        if not keywords: return "无法判断"
        
        match_count = 0
        for k in keywords:
            if k in modern_senses:
                match_count += 1
        
        ratio = match_count / len(keywords)
        if ratio > 0.8: return "古义关键词在现代义项中【高度重合】"
        if ratio > 0.4: return "古义关键词在现代义项中【部分重合】"
        return "古义关键词在现代义项中【几乎未出现】"

    def extract_modern_flags(modern_str):
        if pd.isna(modern_str) or modern_str == "": return "无现代义项"
        s = str(modern_str)
        flags = []
        # 扩大标签捕捉范围
        tags = ["<书>", "〈书〉", "（书）", "[书]", "书面语", "<古>", "〈古〉", "方言", "成语"]
        for t in tags:
            if t in s:
                flags.append(f"含{t}标记")
                break # 只要有一个就行
        return "；".join(flags) if flags else "无限制性标签"

    def compute_decay_feature(row):
        dynasties = ["先秦", "漢", "魏晉南北朝", "唐", "宋", "元", "明", "清", "民國"]
        freqs = [row.get(f"{d}_freq", 0) for d in dynasties]
        if all(f == 0 for f in freqs): return "历史零频次"
        
        # 计算重心
        total = sum(freqs)
        weights = list(range(1, 10)) # 简单的加权
        weighted_sum = sum(f*w for f, w in zip(freqs, weights))
        center = weighted_sum / total
        
        # 1-4为古，5-9为今
        if center < 4.0: return "重心在古代（强衰退）"
        if center < 6.0: return "重心居中（持续使用）"
        return "重心在近现代（现代常用）"

    def numeric_to_text(df):
        
        # 应用新特征
        df["decay_pattern"] = df.apply(compute_decay_feature, axis=1)
        df["modern_flags"] = df["modern_sense_list"].apply(extract_modern_flags)
        df["match_status"] = df.apply(compute_keyword_match, axis=1) # 新增
        df["rarity"] = df.apply(lambda r: "生僻词" if r.get("all_freq", 0) < 20 else "常用词", axis=1)
        return df

    # ... (后续应用逻辑不变)
    train_df["modern_sense_formatted"] = train_df["modern_sense_list"].apply(format_modern_sense)
    train_df = numeric_to_text(train_df)
    
    if test_df is not None:
        test_df["modern_sense_formatted"] = test_df["modern_sense_list"].apply(format_modern_sense)
        test_df = numeric_to_text(test_df)
        return train_df, test_df

    return train_df

### 2. Few-shot 示例采样（增强版）
#### 采样策略
1. 优先从高置信样本（all_freq ≥ 20）中选择
2. 正负样本（label=1/0）均衡采样，每类最多取指定数量（默认3个）
3. 若某类样本不足，取该类全部样本；极端情况随机选1-2条兜底

#### 输入输出
- 输入：预处理后的训练集 DataFrame
- 输出：包含均衡正负例的示例 DataFrame

#### 核心目的
为 Prompt 提供高质量、代表性的参考案例，提升 LLM 分类准确性

In [ ]:
# -------------------------- 2. 策略性示例采样（增强版） --------------------------
def select_few_shot_examples(df, n_per_class=3):
    """
    改进的 Few-shot 示例选择策略：
    - 仅从高置信样本（all_freq >= 20）中采样
    - 每类（label=0 和 label=1）均衡采样 n_per_class 个
    - 若某类不足，则全量使用该类样本
    """
    # 安全获取 all_freq 列，若不存在则默认为0
    all_freq_series = df.get("all_freq", pd.Series([0] * len(df)))
    confident_df = df[all_freq_series >= 20].copy()

    if len(confident_df) == 0:
        print("⚠️ 无高置信样本（all_freq >= 20），回退到全量数据")
        confident_df = df.copy()

    # 分离正负样本
    pos_samples = confident_df[confident_df["label"] == 1]
    neg_samples = confident_df[confident_df["label"] == 0]

    # 每类采样 n_per_class 个（若不足则取全部）
    n_pos = min(n_per_class, len(pos_samples))
    n_neg = min(n_per_class, len(neg_samples))

    selected_pos = pos_samples.sample(n=n_pos, random_state=42) if n_pos > 0 else pd.DataFrame()
    selected_neg = neg_samples.sample(n=n_neg, random_state=42) if n_neg > 0 else pd.DataFrame()

    # 合并并重置索引
    selected = pd.concat([selected_pos, selected_neg], ignore_index=True)

    # 如果仍为空（极端情况），随机选1-2条兜底
    if len(selected) == 0:
        selected = df.sample(n=min(2, len(df)), random_state=42).reset_index(drop=True)

    return selected

# 假设已经加载了 train_df（来自上一步）
selected_examples = select_few_shot_examples(train_df)  

# 现在 selected_examples 是一个 DataFrame，可以在外部使用
print("✅ 成功选出 few-shot 示例：")
for i, row in selected_examples.iterrows():
    print(f"\n--- 示例 {i+1} ---")
    print(f"词: {row['sense']}")
    print(f"标签: {'古今异义' if row['label']==1 else '非古今异义'}")
    print(f"现代义项: {row['modern_sense_formatted']}")
    print(f"标记: {row['modern_flags']}")

### 3. Prompt 构造
#### 构造逻辑
1. 系统提示词：定义语言学家角色，明确古今异义判定的三步核心规则：
   - 第一步：语义匹配（是否有完全一致的现代义项）
   - 第二步：标签检查（有<书>/<古>标记则判定为古今异义）
   - 第三步：频率验证（高频转低频也判定为古今异义）
2. 示例部分：将采样的示例按固定格式拼接（词语/古义/现代义项/标签/逻辑参考）
3. 用户提示词：传入待预测样本的 ID/古义/现代义项/所有辅助特征

#### 输出格式
返回 OpenAI 客户端兼容的 messages 列表（system + user 角色），强制模型输出 JSON 格式结果

In [ ]:
# -------------------------- 3. Prompt 构造模块 --------------------------
def build_prompt(few_shot_examples, sample_id, target_sense, target_modern_senses, target_features):
    
    system_prompt = """你是一个严谨的语言学家，执行“古今异义”二分类判别任务。

【核心判定法则】
判断目标义项（古义）在现代汉语中是否属于“古今异义”（Label 1）还是“义项延续”（Label 0）。

请严格遵守以下逻辑树进行判断：

1. **第一步：语义匹配**
   - 在《现代义项列表》中，能否找到与《目标古义》**含义完全一致**的义项？
   - 如果完全找不到对应含义 -> **直接判定 Label 1 (义项消亡)**。

2. **第二步：标签与语体检查 (至关重要)**
   - 如果找到了含义一致的义项，检查该现代义项是否有**限制性标签**（如 <书>、<古>、<方>、书面语）。
   - **规则**：即使含义一样，如果现代义项被打上了 <书> 或 <古> 的标签，说明它已退出日常口语，属于古今异义 -> **判定 Label 1**。
   - **规则**：如果该义项仅存在于“成语”或“固定搭配”中 -> **判定 Label 1**。

3. **第三步：频率与通用性验证**
   - 如果含义一致且无特殊标签，检查词汇的历史频率趋势。
   - 如果是从古代的高频词变为现代极低频的生僻词 -> **倾向于 Label 1**。
   - 否则 -> **判定 Label 0**。

【输出格式】
必须输出标准的 JSON 格式，包含 "id", "reasoning" (简短分析步骤), "label" (0或1)。
"""

    # 动态构建 Few-Shot 文本（精简版，强调逻辑）
    example_text = ""
    for idx, row in few_shot_examples.iterrows():
        l = row['label']
        # 挑选几个高质量的 reasoning 模板
        if l == 1 and "书" in str(row.get('modern_flags')):
            reason = "虽然现代义项中有匹配的含义，但标记为<书>，属于书面语/文言残留。"
        elif l == 1:
            reason = "现代义项列表中无此含义，语义已转移或消亡。"
        else:
            reason = "现代义项中有完全匹配的含义，且为常用义，无限制性标签。"
            
        example_text += f"""
样本：
- 词语：{row['word']}
- 目标古义：{row['sense']}
- 现代义项：{row['modern_sense_formatted']}
- 辅助特征：{row.get('modern_flags', '')}, {row.get('match_status', '')}
- 实际标签：{l}
- 逻辑参考：{reason}
"""

    user_prompt = f"""
请分析以下待测样本：
样本 ID：{sample_id}
【输入数据】
- 目标词：{target_sense} (这是古籍中的含义)
- 现代义项列表：{target_modern_senses}
- 辅助特征1（字面重合度）：{target_features.get('match_status', '未知')}
- 辅助特征2（标签）：{target_features.get('modern_flags', '未知')}
- 辅助特征3（频率趋势）：{target_features.get('decay_pattern', '未知')}

参考案例：
{example_text}

请输出 JSON：
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

### 4. LLM 预测模块
#### 核心组件
1. 客户端初始化：
   - 从环境变量读取 DASHSCOPE_API_KEY，避免硬编码
   - 配置通义千问兼容 OpenAI 的接口地址
2. 单样本预测：
   - 构造 Prompt，调用 qwen-plus 模型（temperature=0 降低随机性）
   - 重试机制：最多重试2次，失败则默认返回 label=1（保守策略）
   - 结果校验：验证返回 ID 匹配、标签为0/1，异常则修正
3. 批量预测：
   - 遍历测试集，逐个调用单样本预测函数
   - 保存预测结果到 CSV 文件，仅含 id/label 两列

#### 关键参数
- 模型：qwen-plus（性价比高）
- 温度：0（确定性输出）
- 响应格式：强制 JSON_OBJECT，确保解析成功率
- 重试次数：最多2次，每次间隔1秒

In [ ]:
# -------------------------- 4. 异步 LLM 客户端与预测模块--------------------------
def init_llm_client():
    """
    初始化通义千问客户端
    教学点：从环境变量获取 API Key，避免硬编码
    """
    # 从环境变量中读取 API Key
    api_key = os.getenv("DASHSCOPE_API_KEY")
    # Windows Powershell: $env:DASHSCOPE_API_KEY="sk-你的Key"
    # Mac / Linux: export DASHSCOPE_API_KEY="sk-你的Key"
    if not api_key:
        print("错误：未找到 API Key！")
        sys.exit(1)  # 终止程序

    # 初始化客户端
    client = OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )
    
    print("✅ 通义千问客户端初始化成功")
    return client

def predict_single_sample_sync(client, few_shot_examples, sample_id, sample_sense, sample_modern_senses, row, max_retries=2):
    """
    同步预测单个样本的古今异义标签
    """
    # 构造目标特征字典
#     target_features = {
#         "decay_pattern": row.get("decay_pattern", "未知"),
#         "modern_flags": row.get("modern_flags", "未知"),
#         "rarity": row.get("rarity", "未知")
#     }

# 【二次修改了这里】确保所有新特征都放进了 target_features
    target_features = {
        "decay_pattern": row.get("decay_pattern", "未知"),
        "modern_flags": row.get("modern_flags", "未知"),
        "rarity": row.get("rarity", "未知"),
        "match_status": row.get("match_status", "未知") # 记得加上这个！
    }

    # 重试机制：最多尝试 max_retries + 1 次
    for attempt in range(max_retries + 1):
        try:
            messages = build_prompt(
                few_shot_examples=few_shot_examples,
                sample_id=sample_id,
                target_sense=sample_sense,
                target_modern_senses=sample_modern_senses,
                target_features=target_features
            )
            
            print(f"正在预测样本 {sample_id} (第{attempt+1}次尝试)...")
            
            # 发起请求（使用构建好的 messages）
            completion = client.chat.completions.create(
                model="qwen-plus",  # 模型选择：qwen-plus 性价比高
                messages=messages,  # 👈 这里使用 build_prompt 返回的 messages
                temperature=0,      # 降低随机性，提高一致性
                stream=False,       # 非流式输出（结构化输出通常不需要流式）
                # 告诉模型：只返回 JSON
                response_format={"type": "json_object"}
            )

            # 此时拿到的 content 是一个纯净的 JSON 字符串
            json_string = completion.choices[0].message.content
            
            print(f"\n--- 模型原始返回 (字符串) ---")
            print(json_string)

            # 解析 JSON 字符串
            result = json.loads(json_string)
            returned_id = result.get("id")
            label = int(result.get("label", 0))

            # 验证返回结果
            if returned_id != sample_id:
                print(f"⚠️  警告：样本{sample_id}返回id={returned_id}不匹配，强制修正。")
            if label not in [0, 1]:
                raise ValueError(f"标签值 {label} 不在 [0,1] 范围内")

            print(f"✅ 样本 {sample_id} 预测成功：label={label}")
            return sample_id, label

        except json.JSONDecodeError as e:
            print(f"❌ 样本{sample_id}第{attempt+1}次预测失败：JSON解析错误 - {e}")
        except ValueError as e:
            print(f"❌ 样本{sample_id}第{attempt+1}次预测失败：{e}")
        except Exception as e:
            print(f"❌ 样本{sample_id}第{attempt+1}次预测失败：{e}")
        
        # 如果不是最后一次尝试，等待1秒后重试
        if attempt < max_retries:
            print(f"⏳ 等待1秒后重试...")
            time.sleep(1)
        else:
            print(f"❌ 样本{sample_id}最终失败，使用默认label=1（保守策略）")
            return sample_id, 1

    # 如果所有重试都失败，返回默认值
    return sample_id, 1

def batch_predict_and_save(train_path, test_path, output_csv):
    """
    批量预测并保存结果
    """
    # 加载训练集和测试集
    train_df = load_and_preprocess_data(train_path=train_path)
    test_df, _ = load_and_preprocess_data(train_path=train_path, test_path=test_path)
    
    # 选择few-shot示例
    few_shot_examples = select_few_shot_examples(train_df)
    
    # 初始化客户端
    client = init_llm_client()
    
    print(f"🚀 开始批量预测测试集 ({len(test_df)} 个样本)...")
    
    results = []
    for i, (_, row) in enumerate(test_df.iterrows(), 1):
        sample_id = int(row["id"])
        sample_sense = row["sense"]
        sample_modern_senses = row["modern_sense_formatted"]
        
        result = predict_single_sample_sync(
            client=client,
            few_shot_examples=few_shot_examples,
            sample_id=sample_id,
            sample_sense=sample_sense,
            sample_modern_senses=sample_modern_senses,
            row=row
        )
        results.append(result)
        
        if i % 5 == 0 or i == len(test_df):  # 每5个或最后一批显示进度
            print(f"已处理 {i}/{len(test_df)} 个样本")
    
    # 整理结果
    result_df = pd.DataFrame(results, columns=["id", "label"])
    
    # 保存结果
    result_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"✅ 预测结果已保存至: {output_csv}")
    
    return result_df

print("✅ 同步LLM客户端与预测模块已定义")

### 5. 验证评估模块
#### 评估流程
1. 数据拆分：将训练集按比例（默认20%）拆分为演示池（生成示例）和验证集（评估）
2. 特征复用：对演示池/验证集应用简化版预处理，添加核心特征
3. 预测验证：用演示池生成示例，对验证集全量预测
4. 指标计算：
   - 准确率（Accuracy）
   - 分类报告（精确率/召回率/F1）
   - 混淆矩阵（TP/FP/TN/FN）
5. 结果输出：
   - 打印评估指标和预测统计
   - 保存错误样本到 validation_errors.csv

#### 核心目的
在无标注测试集时，验证模型在训练集上的分类效果，定位错误样本

In [ ]:
# -------------------------- 5. 验证评估模块 --------------------------
def evaluate_on_validation(train_path, val_ratio=0.2, random_state=42):
    full_train_df = pd.read_excel(train_path, engine="openpyxl")
    print(f"原始训练集大小: {len(full_train_df)}")

    val_df = full_train_df.sample(frac=val_ratio, random_state=random_state)
    demo_pool = full_train_df.drop(val_df.index).reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(f"演示池大小: {len(demo_pool)}, 验证集大小: {len(val_df)}")

    # 复用预处理逻辑（简化版）
    def add_features(df):
        df = df.copy()
        df["modern_sense_formatted"] = df["modern_sense_list"].apply(
            lambda x: "无现代汉语义项记录" if pd.isna(x) or x == "" else str(x)
        )
        df["modern_flags"] = df["modern_sense_list"].apply(
            lambda s: "含书面语标记" if pd.notna(s) and ("[语体:书]" in str(s) or "〈书〉" in str(s)) else "无特殊标记"
        )
        dynasties = ["先秦", "漢", "魏晉南北朝", "唐", "宋", "元", "明", "清", "民國"]
        def decay(row):
            freqs = [row.get(f"{d}_freq", 0) for d in dynasties]
            early = sum(freqs[:3]); late = sum(freqs[-3:])
            if early > 0 and late == 0: return "仅古代使用"
            elif early > late * 2: return "古代显著高于近代"
            else: return "其他"
        df["decay_pattern"] = df.apply(decay, axis=1)
        df["rarity"] = df.apply(lambda r: "总频次极低（<10）" if r.get("all_freq", 0) < 10 else "总频次正常", axis=1)
        return df

    demo_pool = add_features(demo_pool)
    val_df = add_features(val_df)

    few_shot_examples = select_few_shot_examples(demo_pool)
    client = init_llm_client()

    preds, labels = [], []
    total = len(val_df)

    for i, (_, row) in enumerate(val_df.iterrows(), 1):
        sample_id = int(row["id"])
        sense = row["sense"]
        modern_senses = row["modern_sense_formatted"]
        true_label = int(row["label"])

        print(f"[Val {i}/{total}] 预测样本 ID={sample_id}...")
        _, pred_label = predict_single_sample_sync(  # 使用同步版本
            client=client,
            few_shot_examples=few_shot_examples,
            sample_id=sample_id,
            sample_sense=sense,
            sample_modern_senses=modern_senses,
            row=row
        )
        preds.append(pred_label)
        labels.append(true_label)

        if i < total:
            time.sleep(0.6)

    acc = accuracy_score(labels, preds)
    report = classification_report(labels, preds, target_names=["非古今异义", "古今异义"], digits=4, output_dict=False)

    print("\n" + "="*50)
    print("📊 验证集评估结果")
    print("="*50)
    print(f"准确率 (Accuracy): {acc:.4f}")
    print("\n详细报告:")
    print(report)

    # 显示预测详情
    print("\n" + "="*50)
    print("📋 预测详情")
    print("="*50)
    print(f"总样本数: {len(val_df)}")
    print(f"预测为古今异义数量: {sum(preds)}")
    print(f"预测为非古今异义数量: {len(preds) - sum(preds)}")
    print(f"真实为古今异义数量: {sum(labels)}")
    print(f"真实为非古今异义数量: {len(labels) - sum(labels)}")
    
    # 计算混淆矩阵信息
    true_pos = sum([1 for i in range(len(preds)) if preds[i] == 1 and labels[i] == 1])  # 预测为1，实际为1
    false_pos = sum([1 for i in range(len(preds)) if preds[i] == 1 and labels[i] == 0])  # 预测为1，实际为0
    true_neg = sum([1 for i in range(len(preds)) if preds[i] == 0 and labels[i] == 0])  # 预测为0，实际为0
    false_neg = sum([1 for i in range(len(preds)) if preds[i] == 0 and labels[i] == 1])  # 预测为0，实际为1
    
    print(f"\n混淆矩阵:")
    print(f"真阳(TP, 1→1): {true_pos}, 假阳(FP, 0→1): {false_pos}")
    print(f"真阴(TN, 0→0): {true_neg}, 假阴(FN, 1→0): {false_neg}")

    errors = val_df.copy()
    errors["pred"] = preds
    errors = errors[errors["label"] != errors["pred"]]
    errors.to_csv("validation_errors.csv", index=False, encoding="utf-8-sig")
    print(f"\n❌ 错误样本已保存至: validation_errors.csv ({len(errors)} 条)")

    # 强制刷新输出缓冲区（关键修复！）
    sys.stdout.flush()
    
    return acc, report

print("✅ 验证评估模块已定义")

# 调用验证评估函数（使用清洗后的训练集）
acc, report = evaluate_on_validation("./train_cleaned.xlsx", val_ratio=0.2)

### 6. 测试集预测模块
#### 完整流程
1. 数据加载：加载训练集（生成示例）和测试集，校验测试集必需列
2. 预处理：对测试集应用与训练集一致的特征工程（格式/标签/衰减/生僻度）
3. 示例采样：从训练集生成 Few-shot 示例
4. 批量预测：遍历测试集，调用单样本预测函数，记录 id/label
5. 结果保存：
   - 输出仅含 id/label 的 CSV 文件（UTF-8-SIG 编码，兼容 Excel）
   - 打印预测统计（总样本数/正负例数量）
   - 预览前10行结果

#### 关键输出
- 预测结果文件：./test_predict_yecongrong.csv
- 输出列：id（样本ID）、label（0=非古今异义，1=古今异义）

In [ ]:
# -------------------------- 6. 测试集预测模块 --------------------------

def predict_test_set(train_path, test_path, output_csv, verbose=False):
    """
    使用训练好的模型对测试集进行预测并输出结果

    Args:
        train_path (str): 训练集路径（用于生成 few-shot 示例）
        test_path (str): 测试集路径
        output_csv (str): 预测结果保存路径
        verbose (bool): 是否打印预测过程日志
    """
    print(f"🚀 开始加载训练集与测试集...")
    
    # 加载并预处理训练集（用于构建 few-shot 示例）
    train_df = load_and_preprocess_data(train_path=train_path)
    print(f"✅ 训练集加载完成，共 {len(train_df)} 条数据")
    
    # 加载测试集
    if not os.path.exists(test_path):
        print(f"❌ 找不到测试集文件: {test_path}")
        return None
    
    test_df = pd.read_excel(test_path, engine="openpyxl")
    print(f"✅ 测试集加载完成，共 {len(test_df)} 条数据")
    
    # 检查必要字段
    required_cols = ["id", "sense", "modern_sense_list"]
    for col in required_cols:
        if col not in test_df.columns:
            print(f"❌ 测试集缺失必要字段: {col}")
            return None
    
    # 应用预处理（复用训练集的预处理逻辑）
    test_df["modern_sense_formatted"] = test_df["modern_sense_list"].apply(
        lambda x: "无现代汉语义项记录" if pd.isna(x) or x == "" else str(x)
    )
    test_df["modern_flags"] = test_df["modern_sense_list"].apply(
        lambda s: "含书面语标记" if pd.notna(s) and ("[语体:书]" in str(s) or "〈书〉" in str(s)) else "无特殊标记"
    )
    
    # 生成辅助特征（复用特征工程逻辑）
    dynasties = ["先秦", "漢", "魏晉南北朝", "唐", "宋", "元", "明", "清", "民國"]
    def decay(row):
        freqs = [row.get(f"{d}_freq", 0) for d in dynasties]
        early = sum(freqs[:3]); late = sum(freqs[-3:])
        if early > 0 and late == 0: return "仅古代使用"
        elif early > late * 2: return "古代显著高于近代"
        else: return "其他"
    
    test_df["decay_pattern"] = test_df.apply(decay, axis=1)
    test_df["rarity"] = test_df.apply(
        lambda r: "总频次极低（<10）" if r.get("all_freq", 0) < 10 else "总频次正常", axis=1
    )
    
    print(f"✅ 测试集预处理完成")
    
    # 选择 few-shot 示例（从训练集中采样）
    few_shot_examples = select_few_shot_examples(train_df)
    print(f"✅ 已选出 {len(few_shot_examples)} 个 few-shot 示例")
    
    # 初始化 LLM 客户端
    client = init_llm_client()
    
    print(f"🚀 开始预测测试集...")
    
    results = []
    total = len(test_df)
    
    for i, (_, row) in enumerate(test_df.iterrows(), 1):
        sample_id = int(row["id"])
        sample_sense = row["sense"]
        sample_modern_senses = row["modern_sense_formatted"]
        
        # 预测
        _, pred_label = predict_single_sample_sync(
            client=client,
            few_shot_examples=few_shot_examples,
            sample_id=sample_id,
            sample_sense=sample_sense,
            sample_modern_senses=sample_modern_senses,
            row=row,
        )
        
        # 仅保存 id 和 label 两列
        results.append({"id": sample_id, "label": pred_label})
        
        # 进度提示
        if i % 10 == 0 or i == total:
            print(f"已预测 {i}/{total} 个样本")
    
    # 保存结果（仅 id 和 label 两列）
    result_df = pd.DataFrame(results, columns=["id", "label"])  # 明确指定列顺序
    result_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"✅ 测试集预测完成！结果已保存至: {output_csv}")
    
    # 打印结果统计
    print(f"\n📊 预测结果统计:")
    print(f"  - 总样本数: {len(result_df)}")
    print(f"  - 预测为古今异义 (label=1): {sum(result_df['label'])}")
    print(f"  - 预测为非古今异义 (label=0): {len(result_df) - sum(result_df['label'])}")
    
    # 预览前几行结果
    print(f"\n📋 输出文件预览:")
    print(result_df.head(10).to_string(index=False))
    
    return result_df


test_result = predict_test_set(
    train_path="./train_cleaned.xlsx",
    test_path="./test.xlsx",  # 替换为你的测试集路径
    output_csv="./test_predict_yecongrong.csv",  # 输出文件名
)